# 01 – Khám phá dữ liệu (EDA): Sketchy & Flickr15K

Notebook chạy trên Kaggle, **chỉ đọc** dữ liệu trong `/kaggle/input` và ghi mọi kết quả vào `OUTPUT_DIR` (mặc định `/kaggle/working/eda_outputs`, hiện ở tab *Output* sau khi *Save Version*).

Cấu trúc dữ liệu kỳ vọng:
```
sketchy/{train,val}/{photo,sketch}/<class>/<file>
Flickr15K/images/<class>/<file>
Flickr15K/sketches/<1..10>/<file>   (+ lst_sketches, list_images.txt, list_sketches.txt)
```
Output: manifest CSV của từng dataset, bảng đếm theo lớp, kiểm tra trùng lớp / rò rỉ cặp sketch–ảnh giữa train và val, thống kê độ phân giải & mật độ nét, hình minh họa, `summary.json`.

## 0. Cấu hình – **sửa đường dẫn ở đây**
Để trống hoặc sai đường dẫn thì notebook sẽ tự tìm thư mục tên `sketchy` / `Flickr15K` trong `/kaggle/input`.

In [ ]:
# ======= NHẬP ĐƯỜNG DẪN =======
SKETCHY_ROOT = "/kaggle/input/sketchy/sketchy"      # thư mục chứa train/ và val/
FLICKR_ROOT  = "/kaggle/input/flickr15k/Flickr15K"  # thư mục chứa images/ và sketches/
OUTPUT_DIR   = "/kaggle/working/eda_outputs"
# ==============================

SAMPLE_PER_GROUP = 1500   # số file / (dataset, split, modality) đem đọc pixel để đo kích thước, mật độ nét
N_WORKERS = 8             # số luồng đọc ảnh
SEED = 42
ZIP_OUTPUT = True         # nén OUTPUT_DIR thành 1 file zip để tải về

In [ ]:
import os, re, json, random, shutil, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option("display.width", 160)
random.seed(SEED); np.random.seed(SEED)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
INPUT_BASE = "/kaggle/input"


def find_dir(path, name_candidates):
    """Return `path` if it exists, else search INPUT_BASE (depth <= 4) for a folder with one of the names."""
    if path and os.path.isdir(path):
        return path
    lowered = {n.lower() for n in name_candidates}
    if os.path.isdir(INPUT_BASE):
        for dp, dns, _ in os.walk(INPUT_BASE):
            if dp[len(INPUT_BASE):].count(os.sep) > 4:
                dns[:] = []
                continue
            for d in dns:
                if d.lower() in lowered:
                    return os.path.join(dp, d)
    return None


SKETCHY_ROOT = find_dir(SKETCHY_ROOT, ["sketchy"])
FLICKR_ROOT = find_dir(FLICKR_ROOT, ["Flickr15K", "flickr15k"])
print("SKETCHY_ROOT:", SKETCHY_ROOT)
print("FLICKR_ROOT :", FLICKR_ROOT)

OUT = Path(OUTPUT_DIR); (OUT / "figures").mkdir(parents=True, exist_ok=True)
SUMMARY = {"sketchy_root": SKETCHY_ROOT, "flickr_root": FLICKR_ROOT}


def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT / "figures" / f"{name}.png", dpi=120, bbox_inches="tight")
    plt.show()

### Cây thư mục (3 tầng đầu) để kiểm tra đường dẫn

In [ ]:
def show_tree(root, max_depth=3, max_items=8):
    if not root:
        print("(không tìm thấy)"); return
    root = os.path.abspath(root)
    for dp, dns, fns in os.walk(root):
        depth = dp[len(root):].count(os.sep)
        dns.sort()
        if depth >= max_depth:
            dns[:] = []
        ind = "    " * depth
        print(f"{ind}{os.path.basename(dp)}/  ({len(dns)} dirs, {len(fns)} files)")
        if depth < max_depth:
            for f in sorted(fns)[:3]:
                print(f"{ind}    {f}")
        if len(dns) > max_items:
            print(f"{ind}    ... {len(dns) - max_items} more dirs")
            dns[:] = dns[:max_items]

show_tree(SKETCHY_ROOT); print(); show_tree(FLICKR_ROOT)

## 1. Quét file → manifest

In [ ]:
def scan(root):
    """Recursive os.scandir -> DataFrame(relpath, ext, bytes)."""
    rows, stack = [], [root]
    while stack:
        d = stack.pop()
        with os.scandir(d) as it:
            for e in it:
                if e.is_dir(follow_symlinks=False):
                    stack.append(e.path)
                else:
                    ext = os.path.splitext(e.name)[1].lower()
                    if ext in IMG_EXT:
                        rows.append((os.path.relpath(e.path, root).replace(os.sep, "/"), ext, e.stat().st_size))
    return pd.DataFrame(rows, columns=["relpath", "ext", "bytes"]).sort_values("relpath", ignore_index=True)


t = time.time()
sk = scan(SKETCHY_ROOT) if SKETCHY_ROOT else pd.DataFrame(columns=["relpath", "ext", "bytes"])
fl = scan(FLICKR_ROOT) if FLICKR_ROOT else pd.DataFrame(columns=["relpath", "ext", "bytes"])
print(f"scanned in {time.time() - t:.1f}s  | Sketchy: {len(sk):,} files | Flickr15K: {len(fl):,} files")

In [ ]:
# ---- Sketchy: <split>/<modality>/[...]/<class>/<file>
parts = sk["relpath"].str.split("/")
sk["split"] = parts.str[0]
sk["modality"] = parts.str[1]
sk["category"] = parts.apply(lambda p: p[-2] if len(p) >= 4 else "(none)")
sk["subdir"] = parts.apply(lambda p: "/".join(p[2:-2]))          # vd. tx_000000000000 nếu có
sk["stem"] = parts.str[-1].str.rsplit(".", n=1).str[0]
# Quy ước Sketchy: sketch "n02691156_10151-3" vẽ từ ảnh "n02691156_10151"
sk["photo_id"] = np.where(sk["modality"] == "sketch", sk["stem"].str.replace(r"-\d+$", "", regex=True), sk["stem"])

# ---- Flickr15K: images/<class>/<file>, sketches/<group>/<file>
fparts = fl["relpath"].str.split("/")
fl["top"] = fparts.str[0]
fl["modality"] = fl["top"].map({"images": "photo", "sketches": "sketch"}).fillna(fl["top"])
fl["group"] = fparts.apply(lambda p: p[1] if len(p) >= 3 else "(root)")
fl["category"] = np.where(fl["modality"] == "photo", fl["group"], None)
fl["stem"] = fparts.str[-1].str.rsplit(".", n=1).str[0]

display(sk.head()); display(fl.head())
print("Sketchy subdirs giữa modality và class:", sk["subdir"].value_counts().to_dict())
print("Extensions Sketchy:", sk.groupby("modality")["ext"].value_counts().to_dict())
print("Extensions Flickr :", fl.groupby("modality")["ext"].value_counts().to_dict())

## 2. Sketchy – số lượng theo split / modality / lớp

In [ ]:
overview = sk.pivot_table(index="split", columns="modality", values="relpath", aggfunc="count", fill_value=0)
overview["n_classes"] = sk.groupby("split")["category"].nunique()
display(overview)

counts = sk.pivot_table(index="category", columns=["split", "modality"], values="relpath",
                        aggfunc="count", fill_value=0)
counts.columns = [f"{s}_{m}" for s, m in counts.columns]
counts = counts.sort_values(counts.columns.tolist(), ascending=False)
counts.to_csv(OUT / "sketchy_class_counts.csv")
display(counts.describe().T.round(1))

classes = {s: set(g["category"]) for s, g in sk.groupby("split")}
split_names = sorted(classes)
for i, a in enumerate(split_names):
    for b in split_names[i + 1:]:
        inter = classes[a] & classes[b]
        print(f"Lớp chung {a} ∩ {b}: {len(inter)} / {len(classes[a] | classes[b])}")
all_classes = sorted(set(sk["category"]))
print(f"Tổng số lớp: {len(all_classes)}")
SUMMARY["sketchy"] = {
    "files": int(len(sk)),
    "per_split_modality": {f"{s}/{m}": int(v) for (s, m), v in sk.groupby(["split", "modality"]).size().items()},
    "classes_per_split": {s: len(c) for s, c in classes.items()},
    "n_classes_total": len(all_classes),
}
with open(OUT / "sketchy_classes.txt", "w") as f:
    f.write("\n".join(all_classes))

In [ ]:
for split in split_names:
    sub = counts[[c for c in counts.columns if c.startswith(split + "_")]]
    ax = sub.sort_values(sub.columns[0], ascending=False).plot.bar(figsize=(max(12, len(sub) * 0.18), 4), width=0.85)
    ax.set_title(f"Sketchy {split}: số file theo lớp"); ax.set_xlabel(""); ax.tick_params(axis="x", labelsize=6)
    savefig(f"sketchy_{split}_class_counts")

## 3. Sketchy – ghép cặp sketch ↔ ảnh & kiểm tra rò rỉ giữa train/val
Mỗi ảnh Sketchy có ~5 sketch. Nếu sketch ở `val` nhưng ảnh gốc của nó ở `train` (hoặc ngược lại) thì val **không độc lập** với train. Nếu train và val dùng chung lớp thì đây là split *category-level*, **không phải zero-shot** – cần tự tạo split 100 seen / 25 unseen.

In [ ]:
photos = sk[sk.modality == "photo"]
sketches = sk[sk.modality == "sketch"]
photo_split = photos.groupby("photo_id")["split"].agg(lambda s: ",".join(sorted(set(s))))

pair = sketches[["split", "category", "photo_id"]].copy()
pair["photo_split"] = pair["photo_id"].map(photo_split).fillna("(missing)")
pair["status"] = np.select(
    [pair["photo_split"] == "(missing)", pair["photo_split"] == pair["split"]],
    ["photo không tồn tại", "cùng split"], default="KHÁC split (rò rỉ)")
pair_tab = pd.crosstab(pair["split"], pair["status"])
display(pair_tab)

spp = sketches.groupby("photo_id").size()
print(f"Số sketch / ảnh: mean={spp.mean():.2f}, min={spp.min()}, max={spp.max()}")
print("Ảnh không có sketch nào:", int((~photos["photo_id"].isin(sketches["photo_id"])).sum()))
dup_photo = photos.groupby("photo_id")["split"].nunique()
print("photo_id xuất hiện ở >1 split:", int((dup_photo > 1).sum()))

SUMMARY["sketchy"]["pairing"] = {k: {kk: int(vv) for kk, vv in v.items()} for k, v in pair_tab.to_dict("index").items()}
SUMMARY["sketchy"]["sketches_per_photo_mean"] = float(spp.mean())
SUMMARY["sketchy"]["photo_ids_in_multiple_splits"] = int((dup_photo > 1).sum())

spp.value_counts().sort_index().plot.bar(figsize=(6, 3), title="Sketchy: số sketch mỗi ảnh")
savefig("sketchy_sketches_per_photo")

## 4. Flickr15K – ảnh theo lớp, sketch theo nhóm, file danh sách

In [ ]:
fl_overview = fl.groupby(["modality", "group"]).size().rename("files").reset_index()
print(f"Photo: {int((fl.modality == 'photo').sum()):,} file / {fl.loc[fl.modality == 'photo', 'group'].nunique()} lớp")
print(f"Sketch: {int((fl.modality == 'sketch').sum()):,} file / {fl.loc[fl.modality == 'sketch', 'group'].nunique()} nhóm")
fl_photo_counts = fl[fl.modality == "photo"]["category"].value_counts()
fl_photo_counts.to_csv(OUT / "flickr15k_photo_class_counts.csv", header=["n_photos"])
display(fl_photo_counts.describe().round(1))
display(fl[fl.modality == "sketch"].groupby("group").agg(n=("stem", "size"), examples=("stem", lambda s: list(s[:5]))))

ax = fl_photo_counts.plot.bar(figsize=(12, 4), title="Flickr15K: số ảnh theo lớp")
ax.tick_params(axis="x", labelsize=7)
savefig("flickr15k_photo_class_counts")

In [ ]:
# Các file danh sách đi kèm: in vài dòng đầu để xác định định dạng & ánh xạ sketch -> lớp
list_files = []
if FLICKR_ROOT:
    for dp, _, fns in os.walk(FLICKR_ROOT):
        for f in fns:
            if f.lower().startswith(("lst", "list")) or f.lower().endswith(".txt"):
                list_files.append(os.path.join(dp, f))

list_info = {}
for p in sorted(list_files):
    with open(p, encoding="utf-8", errors="replace") as fh:
        lines = [l.rstrip("\n") for l in fh]
    rel = os.path.relpath(p, FLICKR_ROOT)
    list_info[rel] = {"n_lines": len(lines), "head": lines[:8]}
    print(f"--- {rel}  ({len(lines)} dòng)")
    for l in lines[:8]:
        print("   ", l)
SUMMARY["flickr15k"] = {
    "files": int(len(fl)),
    "n_photos": int((fl.modality == "photo").sum()),
    "n_sketches": int((fl.modality == "sketch").sum()),
    "n_photo_classes": int(fl.loc[fl.modality == "photo", "group"].nunique()),
    "sketch_groups": sorted(fl.loc[fl.modality == "sketch", "group"].unique().tolist()),
    "list_files": list_info,
}

In [ ]:
# Thử ánh xạ sketch -> lớp từ list_sketches.txt: tìm dòng có chứa tên file sketch và một tên lớp ảnh.
photo_classes = set(fl.loc[fl.modality == "photo", "category"])
sketch_map = {}
for rel, info in list_info.items():
    if "sketch" not in rel.lower():
        continue
    with open(os.path.join(FLICKR_ROOT, rel), encoding="utf-8", errors="replace") as fh:
        for line in fh:
            toks = [t for t in re.split(r"[\s,;]+", line.strip().replace("\\", "/")) if t]
            cls = next((t for t in toks if t in photo_classes), None)
            if not cls:
                cls = next((p for t in toks for p in t.split("/") if p in photo_classes), None)
            if cls:
                for t in toks:
                    key = re.sub(r"^(\./)?sketches/", "", t).rsplit(".", 1)[0]
                    sketch_map.setdefault(key, cls)

fl_sk_rows = fl.modality == "sketch"
by_path = (fl.loc[fl_sk_rows, "group"] + "/" + fl.loc[fl_sk_rows, "stem"]).map(sketch_map)
fl.loc[fl_sk_rows, "category"] = by_path.fillna(fl.loc[fl_sk_rows, "stem"].map(sketch_map))
n_mapped = int(fl.loc[fl.modality == "sketch", "category"].notna().sum())
print(f"Sketch có nhãn lớp suy ra từ file danh sách: {n_mapped} / {int((fl.modality == 'sketch').sum())}")
if n_mapped == 0:
    print("-> Chưa suy ra được nhãn; xem định dạng file danh sách ở trên rồi bổ sung luật ánh xạ.")
else:
    display(fl[fl.modality == "sketch"]["category"].value_counts().to_frame("n_sketches"))
SUMMARY["flickr15k"]["sketches_with_class"] = n_mapped

## 5. Thống kê pixel (lấy mẫu): độ phân giải, chế độ màu, mật độ nét

In [ ]:
def img_stats(path):
    try:
        with Image.open(path) as im:
            w, h, mode = im.width, im.height, im.mode
            g = np.asarray(im.convert("L").resize((128, 128)), dtype=np.uint8)
        return w, h, mode, float(g.mean()), float((g < 128).mean())
    except Exception as e:
        return None, None, f"ERROR:{type(e).__name__}", None, None


def sample_stats(df, root, keys, name):
    parts = []
    for key, g in df.groupby(keys):
        parts.append(g.sample(min(len(g), SAMPLE_PER_GROUP), random_state=SEED))
    s = pd.concat(parts, ignore_index=True)
    with ThreadPoolExecutor(N_WORKERS) as ex:
        res = list(ex.map(lambda r: img_stats(os.path.join(root, r)), s["relpath"]))
    s[["width", "height", "mode", "mean_gray", "dark_ratio"]] = pd.DataFrame(res)
    s["dataset"] = name
    return s


t = time.time()
stats = []
if len(sk): stats.append(sample_stats(sk, SKETCHY_ROOT, ["split", "modality"], "sketchy"))
if len(fl): stats.append(sample_stats(fl, FLICKR_ROOT, ["modality"], "flickr15k"))
stats = pd.concat(stats, ignore_index=True)
stats["split"] = stats["split"].fillna("all")
print(f"đọc {len(stats):,} ảnh trong {time.time() - t:.1f}s")
stats.to_csv(OUT / "pixel_stats_sample.csv", index=False)

errors = stats[stats["mode"].astype(str).str.startswith("ERROR")]
print("File lỗi/không mở được:", len(errors))
res_tab = (stats.groupby(["dataset", "split", "modality"])
           .agg(n=("relpath", "size"), w_min=("width", "min"), w_med=("width", "median"), w_max=("width", "max"),
                h_min=("height", "min"), h_med=("height", "median"), h_max=("height", "max"),
                modes=("mode", lambda m: dict(Counter(m))), dark_ratio=("dark_ratio", "mean"),
                mean_gray=("mean_gray", "mean"), kb_mean=("bytes", lambda b: b.mean() / 1024))
           .round(3))
display(res_tab)
res_tab.to_csv(OUT / "pixel_stats_summary.csv")
SUMMARY["pixel_stats"] = json.loads(res_tab.reset_index().to_json(orient="records"))
SUMMARY["unreadable_files_in_sample"] = errors["relpath"].tolist()[:50]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for (ds, mod), g in stats.groupby(["dataset", "modality"]):
    axes[0].scatter(g["width"], g["height"], s=4, alpha=0.4, label=f"{ds}/{mod}")
axes[0].set(title="Độ phân giải (mẫu)", xlabel="width", ylabel="height"); axes[0].legend(markerscale=4)
sk_only = stats[stats.modality == "sketch"]
for ds, g in sk_only.groupby("dataset"):
    axes[1].hist(g["dark_ratio"].dropna(), bins=50, alpha=0.6, label=ds, density=True)
axes[1].set(title="Mật độ nét sketch (tỉ lệ pixel tối)", xlabel="dark ratio"); axes[1].legend()
savefig("resolution_and_stroke_density")

## 6. Ảnh minh họa

In [ ]:
def show_grid(items, ncols, title, name, size=1.8):
    nrows = int(np.ceil(len(items) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size))
    for ax in np.ravel(axes): ax.axis("off")
    for ax, (path, label) in zip(np.ravel(axes), items):
        try:
            ax.imshow(Image.open(path).convert("RGB"))
        except Exception:
            pass
        ax.set_title(label, fontsize=7)
    fig.suptitle(title)
    savefig(name)


# Sketchy: mỗi hàng 1 lớp = 1 ảnh + tối đa 4 sketch vẽ từ chính ảnh đó
rng = random.Random(SEED)
for split in split_names:
    items = []
    cats = sorted(set(photos.loc[photos.split == split, "category"]))
    for c in rng.sample(cats, min(6, len(cats))):
        ph = photos[(photos.split == split) & (photos.category == c)]
        ph = ph[ph.photo_id.isin(sketches.photo_id)]
        if not len(ph):
            continue
        row = ph.sample(1, random_state=SEED).iloc[0]
        sks = sketches[sketches.photo_id == row.photo_id].head(4)
        items.append((os.path.join(SKETCHY_ROOT, row.relpath), f"{c} (photo)"))
        items += [(os.path.join(SKETCHY_ROOT, r), "sketch") for r in sks.relpath]
        items += [(None, "")] * (4 - len(sks))
    if items:
        show_grid(items, 5, f"Sketchy {split}: ảnh và các sketch tương ứng", f"sketchy_{split}_samples")

# Flickr15K: ảnh của 10 lớp ngẫu nhiên + sketch của mỗi nhóm
fl_ph = fl[fl.modality == "photo"]
items = [(os.path.join(FLICKR_ROOT, fl_ph[fl_ph.category == c].sample(1, random_state=SEED).relpath.iloc[0]), c[:18])
         for c in rng.sample(sorted(photo_classes), min(10, len(photo_classes)))]
if items:
    show_grid(items, 5, "Flickr15K: ảnh", "flickr15k_photo_samples")
fl_sk = fl[fl.modality == "sketch"]
items = [(os.path.join(FLICKR_ROOT, r.relpath), f"{r.group}/{r.stem}" + (f"\n{r.category}" if isinstance(r.category, str) else ""))
         for _, r in fl_sk.groupby("group").head(2).head(20).iterrows()]
if items:
    show_grid(items, 5, "Flickr15K: sketch (2 / nhóm)", "flickr15k_sketch_samples")

## 7. Lưu manifest + tóm tắt

In [ ]:
sk.to_csv(OUT / "sketchy_manifest.csv", index=False)
fl.to_csv(OUT / "flickr15k_manifest.csv", index=False)
with open(OUT / "summary.json", "w", encoding="utf-8") as f:
    json.dump(SUMMARY, f, ensure_ascii=False, indent=2, default=str)

print(json.dumps({k: v for k, v in SUMMARY.items() if k != "pixel_stats"}, ensure_ascii=False, indent=2, default=str)[:3000])

if ZIP_OUTPUT:
    zip_path = shutil.make_archive(str(OUT), "zip", OUT)
    print("zip:", zip_path)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1024:9.1f} KB  {p.relative_to(OUT.parent)}")